# Lesson 14: the move into bonds, the fund stays invested in September 2026 at p = 0.099 below t = 0.2

The term spread is the 10-year Treasury yield minus the 3-month bill rate. From the spread of a month, the spread model gives the probability p of a US recession 12 months later, with the weights fitted on the 561 months April 1953 to December 1999. A pension fund moves from shares into bonds when p is at least the threshold t. A move before no recession is a false alarm at a cost of 6, and staying invested before a recession is a miss at a cost of 24. The expected cost of moving is $(1 - p) \cdot 6$ and that of staying $p \cdot 24$, so a move has the lower expected cost from $p = 6/(6 + 24) = 0.2$ on. The decision is taken for September 2026, the last of the 24 months October 2024 to September 2026 whose recession 12 months later is not yet dated.

Treasury yields and recession dates: Federal Reserve Board H.15 and NBER via FRED, Federal Reserve Bank of St. Louis. Snapshot 30 September 2026.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression

months = pd.read_csv("https://digital-ai-finance.github.io/teaching-msc-data-science/data/2026-09-30-us-recession-monthly.csv", index_col="month", parse_dates=True)
train, future = months[months.split == "train"], months[months.split == "unlabelled"]
model = LogisticRegression(C=np.inf).fit(train[["spread"]], train.recession_in_12m)  # C=inf: no penalty
print(f"score z = {model.intercept_[0]:.3f} {model.coef_[0, 0]:+.3f} * spread, fitted on {len(train)} months")
p = model.predict_proba(future[["spread"]])[:, 1]

plt.plot(future.index, p, marker="o", color="black", label="probability p")
plt.axhline(0.2, ls="--", color="grey", label="threshold t = 0.2")
plt.xticks(rotation=30)
plt.xlabel("month (year-month)")
plt.ylabel("probability of a recession 12 months later")
plt.legend(loc="upper right")
plt.show()

In [ ]:
alarm = 6  # cost of a move into bonds before no recession
miss = 24  # cost of staying invested before a recession
t = alarm / (alarm + miss)  # a miss costs four false alarms
p_last = round(p[-1], 3)  # September 2026, rounded to three decimals as on the slide
move, stay = (1 - p_last) * alarm, p_last * miss
print(f"September 2026: probability {p_last} of a recession 12 months later, threshold t = {t:.3f}")
print(f"expected cost of moving {move:.2f}, of staying {stay:.2f}")
print("move into bonds" if stay >= move else "stay invested")
print(f"months from October 2024 above t: {(p >= t).sum()} of {len(p)}")

Change `miss = 24` to `miss = 60` and run the cell again. At a miss cost of 60 the threshold is $t = 6/(6 + 60) = 0.091$, the expected cost of staying is 5.94 against 5.41 for moving, the decision is "move into bonds", and 24 of the 24 months from October 2024 are above t.